# Notebook 01 — Index the ToolBench API Corpus

**Goal:** Pre-compute dense embeddings for all APIs in the ToolBench corpus using OpenAI `text-embedding-3-small`.

This notebook produces two artifacts used by all downstream experiments:
- `corpus_embeddings.npy` — embedding matrix of shape `(N, 1536)`
- `api_names.json` — ordered list of API action names (index-aligned with the embedding matrix)

**Run once; results are cached.**

## Setup

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))
print(f"Repo:      {REPO_ROOT}")
print(f"Toolbench: {TOOLBENCH_DIR}")

## Load the API Corpus

ToolBench organizes APIs hierarchically: **Category → Tool → Endpoint**. We flatten this into a list of individual API entries, each with name, description, category, tool name, and parameters.

In [ ]:
from data.load_toolbench import load_api_corpus
from models.embeddings import format_api_string, get_embeddings
import numpy as np
from tqdm import tqdm

corpus = load_api_corpus(TOOLBENCH_DIR / "toolenv" / "tools")
print(f"Loaded {len(corpus)} APIs across {len(set(a['category'] for a in corpus))} categories")

## Embed All APIs

Each API is formatted as a structured string:
```
[category] > [tool_name]: [name] — [description] | Parameters: [param1, param2, ...]
```
This format preserves the hierarchical structure and parameter information in a single text representation suitable for embedding.

In [ ]:
api_strings = [format_api_string(a) for a in corpus]
print("Sample:", api_strings[0])

# Embed all APIs (cached — safe to re-run)
BATCH = 500
all_embeddings = []
for i in tqdm(range(0, len(api_strings), BATCH)):
    batch = api_strings[i:i+BATCH]
    embs = get_embeddings(batch)
    all_embeddings.append(embs)

corpus_matrix = np.vstack(all_embeddings)
np.save(PROJECT_DIR / "corpus_embeddings.npy", corpus_matrix)
print(f"Saved corpus_embeddings.npy: shape {corpus_matrix.shape}")

## Save API Name Index

Save the ordered list of API action names so downstream notebooks can map embedding indices back to API identities.

In [ ]:
import json
api_names = [a["action_name"] for a in corpus]
with open(PROJECT_DIR / "api_names.json", "w") as f:
    json.dump(api_names, f)
print(f"Saved {len(api_names)} API names")